# Export MIDI-DDSP Decoder to ONNX

This notebook exports the pretrained MIDI-DDSP synthesis decoder to ONNX format.
The exported model maps `(f0_hz, loudness_db, instrument_id)` → DDSP synthesis parameters.

**Instructions:**
1. Run cell 1 (install deps)
2. When prompted, click **Restart runtime** (required for new packages)
3. Then: Runtime → Run all (skip cell 1 on the second run)
4. The ONNX file downloads automatically at the end

In [ ]:
# 1. Install dependencies — RESTART RUNTIME AFTER THIS CELL
!pip install midi-ddsp tf2onnx onnx onnxruntime
print('\n⚠️  Now restart the runtime: Runtime → Restart runtime')
print('Then run all cells starting from cell 2.')

In [ ]:
# 2. Verify imports work after restart
import midi_ddsp
import tensorflow as tf
import tf2onnx
import onnx
import numpy as np
import os
print(f'midi-ddsp: {midi_ddsp.__version__ if hasattr(midi_ddsp, "__version__") else "ok"}')
print(f'tensorflow: {tf.__version__}')
print('All imports OK.')

In [ ]:
# 3. Download pretrained MIDI-DDSP weights
from midi_ddsp.download_model_weights import main as download_weights
download_weights()
print('Weights downloaded.')

In [ ]:
# 4. Load the pretrained synthesis generator
from midi_ddsp.utils.midi_synthesis_utils import load_pretrained_model

synthesis_generator, expression_generator = load_pretrained_model()
print(f'Loaded: {type(synthesis_generator).__name__}')

# Inspect the model
import inspect
print(f'\nModel class: {synthesis_generator.__class__.__name__}')
if hasattr(synthesis_generator, 'call'):
    print(f'call signature: {inspect.signature(synthesis_generator.call)}')

In [ ]:
# 5. Test the model with example inputs to find the right call pattern
N_FRAMES = 250  # 1 second at 250 Hz

# Try multiple input formats until one works
test_cases = [
    ('3D tensors', {
        'args': (
            tf.constant(np.full((1, N_FRAMES, 1), 440.0, dtype=np.float32)),
            tf.constant(np.full((1, N_FRAMES, 1), -30.0, dtype=np.float32)),
            tf.constant([[0]], dtype=tf.int32),
        ),
        'kwargs': {'training': False},
    }),
    ('2D tensors', {
        'args': (
            tf.constant(np.full((1, N_FRAMES), 440.0, dtype=np.float32)),
            tf.constant(np.full((1, N_FRAMES), -30.0, dtype=np.float32)),
            tf.constant([0], dtype=tf.int32),
        ),
        'kwargs': {'training': False},
    }),
    ('dict input', {
        'args': ({
            'f0_hz': tf.constant(np.full((1, N_FRAMES), 440.0, dtype=np.float32)),
            'loudness_db': tf.constant(np.full((1, N_FRAMES), -30.0, dtype=np.float32)),
            'instrument_id': tf.constant([0], dtype=tf.int32),
        },),
        'kwargs': {'training': False},
    }),
]

working_case = None
for name, case in test_cases:
    try:
        outputs = synthesis_generator(*case['args'], **case['kwargs'])
        out_keys = list(outputs.keys()) if isinstance(outputs, dict) else str(type(outputs))
        print(f'✓ {name} works! Output keys: {out_keys}')
        if isinstance(outputs, dict):
            for k, v in outputs.items():
                print(f'  {k}: shape={v.shape}, min={float(tf.reduce_min(v)):.4f}, max={float(tf.reduce_max(v)):.4f}')
        working_case = (name, case)
        break
    except Exception as e:
        print(f'✗ {name}: {e}')

if not working_case:
    raise RuntimeError('Could not find working call signature. Check model API.')

print(f'\nUsing: {working_case[0]}')

In [ ]:
# 6. Export to SavedModel with a concrete function
SAVED_MODEL_DIR = '/tmp/ddsp_synthesis_generator'
case_name, case = working_case

if case_name == '3D tensors':
    @tf.function(input_signature=[
        tf.TensorSpec([1, None, 1], tf.float32, name='f0_hz'),
        tf.TensorSpec([1, None, 1], tf.float32, name='loudness_db'),
        tf.TensorSpec([1, 1], tf.int32, name='instrument_id'),
    ])
    def export_fn(f0_hz, loudness_db, instrument_id):
        out = synthesis_generator(f0_hz, loudness_db, instrument_id, training=False)
        return {k: v for k, v in out.items() if k in ('amplitudes', 'harmonic_distribution', 'noise_magnitudes')}
elif case_name == '2D tensors':
    @tf.function(input_signature=[
        tf.TensorSpec([1, None], tf.float32, name='f0_hz'),
        tf.TensorSpec([1, None], tf.float32, name='loudness_db'),
        tf.TensorSpec([1], tf.int32, name='instrument_id'),
    ])
    def export_fn(f0_hz, loudness_db, instrument_id):
        out = synthesis_generator(f0_hz, loudness_db, instrument_id, training=False)
        return {k: v for k, v in out.items() if k in ('amplitudes', 'harmonic_distribution', 'noise_magnitudes')}
else:  # dict input
    @tf.function(input_signature=[
        tf.TensorSpec([1, None], tf.float32, name='f0_hz'),
        tf.TensorSpec([1, None], tf.float32, name='loudness_db'),
        tf.TensorSpec([1], tf.int32, name='instrument_id'),
    ])
    def export_fn(f0_hz, loudness_db, instrument_id):
        out = synthesis_generator({'f0_hz': f0_hz, 'loudness_db': loudness_db, 'instrument_id': instrument_id}, training=False)
        return {k: v for k, v in out.items() if k in ('amplitudes', 'harmonic_distribution', 'noise_magnitudes')}

# Trace
cf = export_fn.get_concrete_function(*case['args'][:3 if case_name != 'dict input' else 1])
tf.saved_model.save(synthesis_generator, SAVED_MODEL_DIR, signatures={'serving_default': cf})
print(f'SavedModel exported to {SAVED_MODEL_DIR}')

In [ ]:
# 7. Convert SavedModel to ONNX
OUTPUT_PATH = '/tmp/ddsp_decoder.onnx'

model_proto, _ = tf2onnx.convert.from_saved_model(
    SAVED_MODEL_DIR,
    output_path=OUTPUT_PATH,
)

model = onnx.load(OUTPUT_PATH)
onnx.checker.check_model(model)

file_size = os.path.getsize(OUTPUT_PATH)
print(f'\nONNX exported: {OUTPUT_PATH}')
print(f'Size: {file_size / 1024 / 1024:.1f} MB')
print(f'Inputs: {[inp.name for inp in model.graph.input]}')
print(f'Outputs: {[out.name for out in model.graph.output]}')

In [ ]:
# 8. Validate with ONNX Runtime
import onnxruntime as ort

sess = ort.InferenceSession(OUTPUT_PATH)
input_names = [i.name for i in sess.get_inputs()]
output_names = [o.name for o in sess.get_outputs()]
print('Inputs:', input_names)
print('Outputs:', output_names)

# Build feed dict matching whatever input format the model expects
feed = {}
for inp in sess.get_inputs():
    shape = [d if isinstance(d, int) else 250 for d in inp.shape]  # replace dynamic dims with 250
    if 'f0' in inp.name:
        feed[inp.name] = np.full(shape, 440.0, dtype=np.float32)
    elif 'loudness' in inp.name or 'ld' in inp.name:
        feed[inp.name] = np.full(shape, -30.0, dtype=np.float32)
    elif 'instrument' in inp.name:
        feed[inp.name] = np.zeros(shape, dtype=np.int32)
    else:
        feed[inp.name] = np.zeros(shape, dtype=np.float32)

result = sess.run(None, feed)
for i, name in enumerate(output_names):
    r = result[i]
    print(f'{name}: shape={r.shape}, min={r.min():.6f}, max={r.max():.6f}')

# Verify outputs are non-zero
all_nonzero = all(np.abs(r).max() > 1e-6 for r in result)
print(f'\n{"✓" if all_nonzero else "✗"} Validation {"passed" if all_nonzero else "FAILED"} — outputs are {"non-zero" if all_nonzero else "ALL ZEROS (model may not be working)"}')

In [ ]:
# 9. Download
from google.colab import files
files.download(OUTPUT_PATH)

print(f'\n=== Done ===')
print(f'File: ddsp_decoder.onnx ({file_size / 1024 / 1024:.1f} MB)')
print(f'\nUpload to HuggingFace:')
print(f'  hf upload jcosta33/vocoder-models ddsp_decoder.onnx ddsp-decoder/ddsp_decoder.onnx --repo-type model')
print(f'\nUpdate ddspInstrumentCatalog.ts:')
print(f'  export const DDSP_MODEL_SIZE_BYTES = {file_size};')